# 07. Privacy: DP-SGD, PATE, Gradient Inversion (DLG), and Membership Inference (LiRA)

- **Category:** 11_ai_safety_privacy_and_verification
- **Lineage & Citations:** Abadi et al. (2016 *Deep Learning with Differential Privacy*); Papernot et al. (2017 *Semi-Supervised Knowledge Transfer for Deep Learning from Private Training Data*); Zhu et al. (2019 *Deep Leakage from Gradients*); Carlini et al. (2022 *Membership Inference Attacks From First Principles*).
- **Purpose:** Formulate and evaluate differential privacy algorithms and auditing attacks: DP-SGD per-sample gradient clipping and Gaussian mechanism, PATE noisy aggregation, DLG gradient inversion loss, and LiRA likelihood-ratio membership inference.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [06. Backdoor Trojan and Poisoning Defense Neural Cleanse and Spectral](06_backdoor_trojan_and_poisoning_defense_neural_cleanse_and_spectral.ipynb) $\to$ `🔗 Downstream Syntheses:` [08. AI Watermarking AI Detection and Model Obfuscation](08_ai_watermarking_ai_detection_and_model_obfuscation.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do not use batch-level clipping instead of per-sample clipping in DP-SGD (violates privacy sensitivity guarantees); do not evaluate membership inference using naive raw loss thresholds across heterogeneous data samples without parametric shadow model calibration.

## Mathematical Foundations & Governing Equations

1. **DP-SGD Gradient Update**: $\tilde{g} = \frac{1}{B} \left( \sum_{i=1}^B g_i \min\left(1, \frac{C}{\|g_i\|_2 + 10^{-8}}\right) + \sigma C \xi \right), \quad \xi \sim \mathcal{N}(0, I)$
2. **PATE Laplace Consensus**: $\tilde{V} = V + \text{Laplace}\left(0, \frac{1}{\lambda}\right)$
3. **DLG Gradient Inversion Matching**: $\mathcal{L}_{\text{DLG}}(x', y') = \|\nabla_\theta \mathcal{L}(x', y') - g^*\|_2^2$
4. **LiRA Membership Score**: $\Lambda = \frac{\mu_{\text{out}} - \ell_{\text{target}}}{\sigma_{\text{out}} + 10^{-8}}$

In [ ]:
import sys, time
from pathlib import Path
import torch
from safetensors.torch import load_file

r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
if str(r) not in sys.path: sys.path.insert(0, str(r))
w_path = r / "weights/11_ai_safety_privacy_and_verification/07_privacy_dp_sgd_pate_gradient_inversion_dlg_and_mia.safetensors"
weights = load_file(str(w_path))
print(f"Loaded {len(weights)} tensors from {w_path.name}")

In [ ]:
def dp_sgd_clip_and_noise(per_sample_grads, clip_norm, noise_vec, noise_multiplier):
    b = per_sample_grads.shape[0]
    norms = torch.norm(per_sample_grads, p=2, dim=-1, keepdim=True) + 1e-8
    scales = torch.clamp(float(clip_norm) / norms, max=1.0)
    clipped = per_sample_grads * scales
    clipped_sum = torch.sum(clipped, dim=0)
    noisy_sum = clipped_sum + float(noise_multiplier) * float(clip_norm) * noise_vec
    return noisy_sum / float(b)

def pate_noisy_aggregation(teacher_votes, laplace_noise, laplace_scale):
    return teacher_votes + float(laplace_scale) * laplace_noise

def dlg_gradient_inversion_loss(dummy_grad, target_grad):
    diff = dummy_grad - target_grad
    return torch.sum(diff ** 2)

def lira_membership_score(target_loss, shadow_out_losses):
    mu_out = torch.mean(shadow_out_losses, dim=-1)
    var_out = torch.mean((shadow_out_losses - mu_out.unsqueeze(-1)) ** 2, dim=-1)
    sigma_out = torch.sqrt(var_out + 1e-8)
    return (mu_out - target_loss) / sigma_out

In [ ]:
# Parity verification against golden reference tensors
dp_grad = dp_sgd_clip_and_noise(
    weights["per_sample_grads"], clip_norm=1.0, noise_vec=weights["noise_vec"], noise_multiplier=0.5
)
assert torch.max(torch.abs(dp_grad - weights["dp_grad_ref"])) < 1e-5

pate_votes = pate_noisy_aggregation(
    weights["teacher_votes"], weights["laplace_noise"], laplace_scale=0.5
)
assert torch.max(torch.abs(pate_votes - weights["pate_votes_ref"])) < 1e-5

dlg_loss = dlg_gradient_inversion_loss(weights["dummy_grad"], weights["target_grad"])
assert torch.max(torch.abs(dlg_loss - weights["dlg_loss_ref"])) < 1e-5

lira_scores = lira_membership_score(weights["target_loss"], weights["shadow_out_losses"])
assert torch.max(torch.abs(lira_scores - weights["lira_scores_ref"])) < 1e-5
print("All DP-SGD, PATE, DLG, and LiRA parity assertions passed!")

In [ ]:
# Benchmarking privacy and auditing operators
t0 = time.perf_counter()
for _ in range(500):
    _ = dp_sgd_clip_and_noise(weights["per_sample_grads"], 1.0, weights["noise_vec"], 0.5)
    _ = pate_noisy_aggregation(weights["teacher_votes"], weights["laplace_noise"], 0.5)
    _ = dlg_gradient_inversion_loss(weights["dummy_grad"], weights["target_grad"])
    _ = lira_membership_score(weights["target_loss"], weights["shadow_out_losses"])
lat_us = (time.perf_counter() - t0) / 500 * 1e6
print(f"Combined DP-SGD, PATE, DLG, LiRA latency: {lat_us:.2f} us")

In [ ]:
# Failure Mode & Auditing: Membership Inference & Gradient Inversion
# 1. LiRA: samples with low loss relative to shadow distribution have high membership score
for idx, score in enumerate(lira_scores):
    status = "MEMBER (TRAIN)" if score.item() > 1.5 else "NON-MEMBER"
    print(f"Sample {idx}: loss = {weights['target_loss'][idx].item():.2f}, LiRA score = {score.item():.2f} -> {status}")
assert lira_scores[2].item() > 2.0, "Sample 2 has anomalous low loss and must have high membership score!"

# 2. DLG Gradient Matching: loss measures discrepancy between dummy and target gradients
print(f"DLG Gradient matching loss: {dlg_loss.item():.4f}")

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **DP-SGD** bounds sensitivity via per-sample $L_2$ clipping and Gaussian noise injection, preventing gradient reconstruction.
2. **PATE** isolates private partitions with an ensemble of teachers, transferring knowledge to a student through Laplace noise.
3. **DLG** demonstrates that unperturbed gradients leak private training images and labels in centralized or federated architectures.
4. **LiRA** provides empirical privacy auditing by estimating parametric likelihood ratios under calibrated shadow models.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [07_privacy_dp_sgd_pate_gradient_inversion_dlg_and_mia.md](../../markdown/11_ai_safety_privacy_and_verification/07_privacy_dp_sgd_pate_gradient_inversion_dlg_and_mia.md)
- **PyTorch 2.x Module:** [07_privacy_dp_sgd_pate_gradient_inversion_dlg_and_mia.py](../../pytorch2/11_ai_safety_privacy_and_verification/07_privacy_dp_sgd_pate_gradient_inversion_dlg_and_mia.py)
- **Functional JAX Module:** [07_privacy_dp_sgd_pate_gradient_inversion_dlg_and_mia.py](../../jax/11_ai_safety_privacy_and_verification/07_privacy_dp_sgd_pate_gradient_inversion_dlg_and_mia.py)
- **Native Rust Candle Track:** [07_privacy_dp_sgd_pate_gradient_inversion_dlg_and_mia.rs](../../candle/11_ai_safety_privacy_and_verification/07_privacy_dp_sgd_pate_gradient_inversion_dlg_and_mia.rs)
- **Hardware Kernel (CPU / WGSL):** [07_privacy_dp_sgd_pate_gradient_inversion_dlg_and_mia.rs](../../kernels/11_ai_safety_privacy_and_verification/07_privacy_dp_sgd_pate_gradient_inversion_dlg_and_mia.rs)
- **Golden SafeTensors:** [07_privacy_dp_sgd_pate_gradient_inversion_dlg_and_mia.safetensors](../../weights/11_ai_safety_privacy_and_verification/07_privacy_dp_sgd_pate_gradient_inversion_dlg_and_mia.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 06. Backdoor Trojan and Poisoning Defense Neural Cleanse and Spectral](06_backdoor_trojan_and_poisoning_defense_neural_cleanse_and_spectral.ipynb) | [Master Index](../../README.md#11_ai_safety_privacy_and_verification) | [Next: 08. AI Watermarking AI Detection and Model Obfuscation](08_ai_watermarking_ai_detection_and_model_obfuscation.ipynb) &rarr;